# Module 4 — From metrics to meaning

**ODSC AI West 2026 · Production-Grade Agent & LLM Observability with Opik**

Everything so far measures whether the system *ran*: latency, cost, errors, span trees.
All of it can look perfect while the answer is wrong.

This module measures whether it was *right* — offline against a dataset, then online
against production traffic as it arrives.

It ends with a **Your turn** exercise: writing a question that fools the agent, then seeing
whether it fools the judge.

## Setup

In [ ]:
%pip install -q opik openai

In [ ]:
import os, sys

# Work from the repo root. A local Jupyter starts in notebooks/, so step up; Colab starts
# empty, so clone the repo first.
if not os.path.exists("workshop"):
    if os.path.exists("../workshop"):
        os.chdir("..")
    else:
        !git clone -q https://github.com/comet-ml/opik-odsc-west-2026.git
        os.chdir("opik-odsc-west-2026")
sys.path.insert(0, os.getcwd())

import getpass
import opik

PROJECT = "odsc-evaluation"
# Self-hosted Opik? Export OPIK_URL_OVERRIDE before starting Jupyter (see SETUP.md).
# Otherwise this connects to Opik Cloud and asks for your API key.
SELF_HOSTED_URL = os.environ.get("OPIK_URL_OVERRIDE")
opik.configure(use_local=bool(SELF_HOSTED_URL), url_override=SELF_HOSTED_URL, project_name=PROJECT)
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key (Enter to continue without one): ")

os.environ["OPIK_PROJECT_NAME"] = PROJECT
client = opik.Opik()

## 1. The failure that looks like success

Our corpus states that span and trace batches flush every 2.0 seconds. It says nothing
about feedback score batches. Watch what the agent does with that gap.

In [ ]:
from workshop.agent import answer_question

result = answer_question("How often does the SDK flush feedback score batches on a timer?")
print(result["answer"])

It answers **2.0 seconds**, fluently and without hedging. The real value is 1.0 seconds.

HTTP 200. Sub-second. Zero errors. Completely wrong, and nothing in Modules 1-3 would
have told you.

## 2. Offline evaluation

Offline evaluation runs a task over a fixed dataset and scores the results, so you can
compare versions. `data/eval_dataset.jsonl` has 28 questions: answerable ones, three
out-of-scope, and five **adversarial** ones whose answers are *almost* in the corpus.

The adversarial items are the interesting ones. The right behaviour is to decline.

In [ ]:
import json

rows = [json.loads(line) for line in open("data/eval_dataset.jsonl")]
adversarial = [r for r in rows if r["category"] == "adversarial_refusal"]
answerable = [r for r in rows if r["category"] == "answerable"][:5]
cases = adversarial + answerable

dataset = client.get_or_create_dataset(name="odsc-grounding")
dataset.insert([
    {"question": r["question"], "reference": r["reference_answer"], "category": r["category"]}
    for r in cases
])
print(f"{len(cases)} items ({len(adversarial)} adversarial, {len(answerable)} answerable)")

A metric is any object with a `score()` method. This one asks a narrow question: did the
agent decline, or did it invent something?

In [ ]:
from opik.evaluation import evaluate
from opik.evaluation.metrics import base_metric, score_result

REFUSALS = ("does not", "doesn't", "not specify", "not mention",
            "not provide", "no information", "cannot")


class Declined(base_metric.BaseMetric):
    def __init__(self):
        super().__init__(name="declined")

    def score(self, output: str, **ignored):
        hit = any(marker in output.lower() for marker in REFUSALS)
        return score_result.ScoreResult(value=1.0 if hit else 0.0, name=self.name)


def make_task(grounded: bool):
    def task(item):
        result = answer_question(item["question"], grounded=grounded)
        return {"output": result["answer"], "context": result["context"]}
    return task


def by_category(result):
    """Declining is correct on adversarial items and wrong on answerable ones, so a
    single aggregate hides the trade-off — and would reward an agent that declines
    everything."""
    out = {}
    for r in result.test_results:
        category = (r.test_case.dataset_item_content or {}).get("category", "?")
        value = r.score_results[0].value
        hits, total = out.get(category, (0, 0))
        out[category] = (hits + value, total + 1)
    return out

In [ ]:
v1 = evaluate(
    dataset=dataset,
    task=make_task(grounded=False),
    scoring_metrics=[Declined()],
    experiment_name="prompt-v1",
    task_threads=4,
    verbose=0,
)
v1_split = by_category(v1)
for category, (hits, total) in sorted(v1_split.items()):
    print(f"  {category:<22} declined {hits:.0f}/{total}")

## 3. The fix is one line

`workshop/agent.py` carries two system prompts. The default is what a first-draft RAG app
ships with. `GROUNDED` adds a single instruction:

> *Answer using only the context provided. If the context does not contain the answer, say
> so plainly instead of guessing.*

Same dataset, same metric, one sentence different.

In [ ]:
v2 = evaluate(
    dataset=dataset,
    task=make_task(grounded=True),
    scoring_metrics=[Declined()],
    experiment_name="prompt-v2-grounded",
    task_threads=4,
    verbose=0,
)
v2_split = by_category(v2)

print(f"{'category':<22} {'v1':>8} {'v2':>8}")
for category in sorted(v1_split):
    a, total = v1_split[category]
    b, _ = v2_split[category]
    print(f"{category:<22} {a:>5.0f}/{total} {b:>5.0f}/{total}")

print("\n(declining is CORRECT on adversarial items, WRONG on answerable ones)")

Read the two columns in opposite directions. On **adversarial** items declining is the
right answer and should go up. On **answerable** items declining is a regression and
should stay at zero — an agent that refuses everything would score perfectly on the first
column and be useless.

On the adversarial items this moves hallucinations from **4/5 to 1/5**.

A 75% reduction from one sentence — and, more usefully, a number you can put in a PR and
defend. Both runs are experiments in Opik, so the comparison is durable rather than
something you eyeballed once.

Note what it does *not* do: eliminate the problem. One case survives — the feedback-score
flush interval — because the wrong answer genuinely *is* in the context, just attached to a
different thing. No prompt fix closes that. Which is the argument for the next section.

## 4. Online scoring

Offline evaluation tells you how the agent does on questions you thought of. Production
sends questions you didn't.

An **online scoring rule** is an LLM-as-a-judge that runs server-side against production
traces as they arrive, and writes its verdict back onto the trace as a feedback score.

> ### Prerequisite, and it is easy to miss
>
> The judge runs **on the Opik server**, not in this notebook. So Opik needs its own LLM
> provider key — the `OPENAI_API_KEY` above is not visible to it. Add one under
> **Configuration → AI Providers** in the UI, or with the cell below.
>
> Without it the rule fires, logs `API key not configured for LLM`, and silently scores
> nothing.

In [ ]:
# New workspaces already include Opik's built-in free provider, so "any provider configured"
# is not enough: the judge below calls OpenAI, so it needs an OpenAI key specifically.
keys = client.rest_client.llm_provider_key.find_llm_provider_keys()
providers = [k.provider for k in (keys.content or [])]
if "openai" not in providers:
    client.rest_client.llm_provider_key.store_llm_provider_api_key(
        provider="openai", api_key=os.environ["OPENAI_API_KEY"],
    )
    print("stored an OpenAI provider key on the Opik server")
else:
    print(f"OpenAI already configured (providers: {providers})")

### Defining the rule

Three parts worth understanding:

- **`sampling_rate`** — the share of production traces scored. Each evaluation is another
  model call, so this is a real cost line. 1.0 here because we want to see it immediately;
  in production you dial it to whatever gives you signal and raise it when chasing a
  regression.
- **`variables`** — where each `{{placeholder}}` reads from in the trace. Get this wrong and
  the judge silently grades an empty string. (Our agent returns context in its *output*, so
  it is `output.context`, not `input.context` — a mistake worth making once.)
- **`schema_`** — the scores returned, enforced with structured outputs.

If the mapping is fiddly, put `{{trace}}` in the prompt instead: **Agent as a Judge** reads
the trace itself with tools, and needs no mapping at all.

In [ ]:
import time

from opik.rest_api.types import automation_rule_evaluator_write as rules
from opik.rest_api.types.llm_as_judge_code_write import LlmAsJudgeCodeWrite
from opik.rest_api.types.llm_as_judge_message_write import LlmAsJudgeMessageWrite
from opik.rest_api.types.llm_as_judge_model_parameters_write import LlmAsJudgeModelParametersWrite
from opik.rest_api.types.llm_as_judge_output_schema_write import LlmAsJudgeOutputSchemaWrite

from opik.rest_api.core.api_error import ApiError


def wait_for(predicate, timeout=180, interval=5):
    """Ingestion is asynchronous and batched, so a freshly logged trace is not instantly
    queryable — and until the first one lands, its project does not exist yet either.
    Poll instead of guessing at a sleep."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            found = predicate()
        except ApiError as exc:
            if exc.status_code != 404:  # 404: the project hasn't been created yet
                raise
            found = None
        if found:
            return found
        time.sleep(interval)
    raise TimeoutError(f"nothing arrived after {timeout}s")


answer_question("warm up so the project exists")
opik.flush_tracker()
project_id = wait_for(lambda: client.rest_client.projects.retrieve_project(name=PROJECT).id)

PROMPT = (
    "You are auditing whether an answer is grounded in retrieved context.\n\n"
    "Work claim by claim. For each factual claim in the ANSWER, locate the sentence in "
    "CONTEXT that states it. A claim is grounded ONLY if the context asserts it about the "
    "SAME subject. Watch specifically for claims that reuse a number, name or property "
    "that the context states about a DIFFERENT subject — those are NOT grounded, however "
    "plausible they look.\n\n"
    "Return grounded=0 if any claim fails this test, 1 only if all pass.\n\n"
    "CONTEXT:\n{{context}}\n\nQUESTION: {{question}}\n\nANSWER: {{answer}}"
)

evaluators = client.rest_client.automation_rule_evaluators
stale = [r.id for r in (evaluators.find_evaluators(project_id=project_id).content or [])
         if r.name == "grounding-audit"]
if stale:  # re-running this cell must not stack duplicate judges on every trace
    evaluators.delete_automation_rule_evaluator_batch(ids=stale, project_id=project_id)

evaluators.create_automation_rule_evaluator(
    request=rules.AutomationRuleEvaluatorWrite_LlmAsJudge(
        name="grounding-audit",
        project_id=project_id,
        sampling_rate=1.0,
        enabled=True,
        action="evaluator",
        code=LlmAsJudgeCodeWrite(
            model=LlmAsJudgeModelParametersWrite(name="gpt-4o", temperature=0.0),
            messages=[LlmAsJudgeMessageWrite(role="USER", content=PROMPT)],
            variables={
                "context": "output.context",
                "question": "input.question",
                "answer": "output.answer",
            },
            schema_=[LlmAsJudgeOutputSchemaWrite(
                name="grounded", type="INTEGER",
                description="1 if every claim appears in the context, else 0")],
        ),
    )
)
print("rule created — it now grades every new trace in this project")

### Watch it fire

Nothing below calls the judge. These are ordinary agent calls; the scoring happens
server-side, off the request path, after the traces land.

Most runs catch both hallucinations. Occasionally the judge waves one through —
section 5 explains why that happens, and why it is worth measuring.

In [ ]:
import datetime as dt
import textwrap

QUESTIONS = [
    "What licence is Opik released under?",
    "How often does the SDK flush feedback score batches on a timer?",
    "What is the sort order of the spans table?",
]

RUN_START = dt.datetime.now(dt.timezone.utc)  # ignore verdicts from earlier runs of this cell
for q in QUESTIONS:
    answer_question(q)
opik.flush_tracker()
print("3 traces logged; waiting for the judge...\n")

scored, deadline = {}, time.time() + 180
while time.time() < deadline and len(scored) < len(QUESTIONS):
    for t in client.search_traces(project_name=PROJECT, max_results=50):
        question = (t.input or {}).get("question")
        if (t.name == "rag_agent" and question in QUESTIONS
                and t.start_time >= RUN_START and t.feedback_scores):
            scored[question] = t.feedback_scores[0]
    if len(scored) < len(QUESTIONS):
        time.sleep(5)

for q in QUESTIONS:
    fs = scored.get(q)
    if fs:
        print(f"grounded={fs.value:.0f}  {q}")
        print(textwrap.indent(textwrap.fill(fs.reason, 80), " " * 10) + "\n")
    else:
        print(f"(not scored yet) {q}\n")

## 5. Your judge is also an LLM

That prompt and that model were not the first thing tried. Measured on five labelled cases:

| judge configuration | correct |
|---|---|
| plain "is this supported by the context?", gpt-4o-mini | 2 / 4 |
| claim-by-claim prompt, gpt-4o-mini | 3 / 5 |
| claim-by-claim prompt, **gpt-4o** | **4 / 5** |

The naive judge failed in exactly the way the agent did — it accepted "2.0 seconds"
because the context mentions 2.0 seconds, for a different thing.

So a judge is not an oracle you install. It is another LLM component, with its own failure
modes, that needs evaluating against labelled data before you trust its numbers. The
dataset you built in section 2 is exactly what you evaluate it with.

That is the loop closing on itself, and it is the honest version of this story.

## Your turn — fool the agent, then try to fool the judge

Write a question whose answer is *almost* in the corpus: a real number or name that the corpus
states about a *different* thing. Browse `data/corpus/` for ideas.

1. Does the agent hallucinate?
2. Does the online judge catch it? The verdict and its reason land on the trace within a minute
   or so — no code calls the judge.
3. Stretch: ask again with `answer_question(MY_QUESTION, grounded=True)`.

An example question is filled in. Replace it with your own.

In [ ]:
MY_QUESTION = "How often does the SDK flush experiment item batches on a timer?"

my_start = dt.datetime.now(dt.timezone.utc)
my_answer = answer_question(MY_QUESTION)
print(my_answer["answer"])
opik.flush_tracker()

### Wait for the judge's verdict

Either outcome teaches something. Judges vary from run to run: in testing, this example
fooled the agent every time and was caught by the judge five times out of seven. A miss is
not a broken exercise — it is section 5's point, live.

In [ ]:
def my_verdict():
    for t in client.search_traces(project_name=PROJECT, max_results=50):
        if ((t.input or {}).get("question") == MY_QUESTION
                and t.start_time >= my_start and t.feedback_scores):
            return t.feedback_scores[0]
    return None


verdict = wait_for(my_verdict)
print(f"grounded={verdict.value:.0f}")
print(textwrap.fill(verdict.reason, 90))

## What you just did

- Caught a confident, fluent, completely wrong answer that every operational signal called
  a success
- Measured it offline against a dataset, fixed it with one sentence, and measured again
  (**4/5 → 1/5**)
- Found the case no prompt fix closes, and deployed an online judge for it
- Discovered the judge needs calibrating too, and knew that because you had labelled data

**instrument → ingest → analyze → evaluate**, and back to instrument.

Everything here is in the repo. Thanks for coming.

- Opik — https://github.com/comet-ml/opik (Apache-2.0)
- Docs — https://www.comet.com/docs/opik/
- Questions — Comet community Slack, channel `#support-issues`: https://chat.comet.com
- Bugs and feature requests — https://github.com/comet-ml/opik/issues
- Contributions — https://github.com/comet-ml/opik/pulls